In [38]:
pip install pdfplumber

Note: you may need to restart the kernel to use updated packages.


In [39]:
import pdfplumber

pdf_path = "sample_pdfs/sample_pdf_1.pdf"
  # <-- change this to any one PDF file name

with pdfplumber.open(pdf_path) as pdf:
    text = ""
    for page in pdf.pages:
        t = page.extract_text() or ""
        text += t + "\n"

print(text)


Seite 1 von 1
ABC Corporation Bestellung AUFNR34343 im Auftrag von 3498578433
Beispielname Unternehmen
Beispielname Unternehmen · Albertus-Magnus-Str. 8, Matternfeld, SL 44624 Kundenanschrift
Beispielname Unternehmen
medical equipment (Deutschland)
Albertus-Magnus-Str. 8,
Industriestraße 3
Matternfeld, SL 44624
12345 Köln Deutschland
Ihre Faxnummer: 0800-12646711
Bestellung AUFNR34343 vom 22.05.2024
Unsere Kundennummer Unser(e) Einkäufer(in) Telefon für Rückfragen Fax für Rückfragen
11223344 Beispielname 060/1212121 0102860405
Endkundennummer
7654321
Bitte liefern Sie an: Gewünschtes Lieferdatum
Zentraleinkauf sofort
Beispielname Unternehmen
Albertus-Magnus-Str. 8, Lieferbedingungen
Keine Angabe
Matternfeld, SL 44624
Zahlungsbedingungen
0 Tage 2,0% Skonto
Pos. Artikelbeschreibung Preis in Menge Einheit Umrechnung Bestellwert
EUR in EUR
1 Sterilisationsmittel 4 VE 1 VE=20 Stück 64,00
Lief.Art.Nr: 000253655G
Interne Mat.Nr: 49115 16,0000 pro 1 VE
Kostenstelle: 20457
Gesamtwert EUR 64,00


In [40]:
import re
from dateutil.parser import parse as parse_date

# ------------------------------
# Extract Invoice Number
# ------------------------------
def extract_invoice_number(text):
    m = re.search(r"Bestellung\s+([A-Za-z0-9]+)", text, re.IGNORECASE)
    if m:
        return m.group(1)
    return None


# ------------------------------
# Extract Invoice Date
# ------------------------------
def extract_invoice_date(text):
    m = re.search(r"(\d{1,2}[.\-/]\d{1,2}[.\-/]\d{4})", text)
    if m:
        try:
            return str(parse_date(m.group(1)).date())
        except:
            return None
    return None


# ------------------------------
# Convert German number format
# ------------------------------
def extract_number(value):
    try:
        return float(value.replace(".", "").replace(",", "."))
    except:
        return None


# ------------------------------
# Extract NET, TAX, GROSS Total
# ------------------------------
def extract_totals(text):
    # ---------- NET ----------
    net_match = re.search(
        r"Gesamtwert\s*(?:EUR)?\s*([0-9]{1,3}[.,][0-9]{2})",
        text,
        re.IGNORECASE
    )
    net = extract_number(net_match.group(1)) if net_match else None

    # ---------- TAX ----------
    # Find the MwSt position
    mwst = re.search(r"MwSt", text, re.IGNORECASE)
    tax = None

    if mwst:
        # Extract text AFTER "MwSt"
        after = text[mwst.end():]

        # Find ALL monetary values AFTER MwSt
        nums = re.findall(r"([0-9]{1,3}[.,][0-9]{2})", after)

        # Remove the percentage value 19,00 if captured
        nums = [n for n in nums if not n.endswith("00")]  # removes "19,00"

        if nums:
            tax = extract_number(nums[0])  # FIRST valid amount after MwSt

    # ---------- GROSS ----------
    gross_match = re.search(
        r"Gesamtwert inkl\. MwSt\.\s*(?:EUR)?\s*([0-9.,]+)",
        text,
        re.IGNORECASE
    )
    gross = extract_number(gross_match.group(1)) if gross_match else None

    return net, tax, gross



# ------------------------------
# Test function
# ------------------------------
def extract_invoice_fields(text):
    invoice_number = extract_invoice_number(text)
    invoice_date = extract_invoice_date(text)
    net_total, tax_amount, gross_total = extract_totals(text)

    return {
        "invoice_number": invoice_number,
        "invoice_date": invoice_date,
        "net_total": net_total,
        "tax_amount": tax_amount,
        "gross_total": gross_total
    }


In [41]:
def build_invoice_object(pdf_path, text):
    invoice_number = extract_invoice_number(text)
    invoice_date = extract_invoice_date(text)
    net_total, tax_amount, gross_total = extract_totals(text)

    invoice = {
        "invoice_number": invoice_number,
        "invoice_date": invoice_date,
        "seller_name": "Unknown Seller",   # TODO: improve later if needed
        "buyer_name": "Unknown Buyer",     # TODO: improve later
        "currency": "EUR",                 # Most German invoices use EUR
        "net_total": net_total,
        "tax_amount": tax_amount,
        "gross_total": gross_total,
        "source_pdf": pdf_path
    }
    return invoice


# Test it
invoice_data = build_invoice_object(pdf_path, raw_text)
invoice_data


{'invoice_number': 'AUFNR34343',
 'invoice_date': '2024-05-22',
 'seller_name': 'Unknown Seller',
 'buyer_name': 'Unknown Buyer',
 'currency': 'EUR',
 'net_total': 64.0,
 'tax_amount': 12.16,
 'gross_total': 76.16,
 'source_pdf': 'sample_pdfs/sample_pdf_1.pdf'}

In [42]:
import json

def save_invoices_to_json(invoices, output_path="extracted_invoices.json"):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(invoices, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(invoices)} invoices to {output_path}")


# Save your extracted invoices
save_invoices_to_json(all_invoices, "extracted_invoices.json")


Saved 5 invoices to extracted_invoices.json


In [43]:
from dateutil.parser import parse as parse_date

# ------------------------------
# Validate ONE invoice
# ------------------------------
def validate_invoice(inv):
    errors = []

    # ---------- Rule 1: Required fields ----------
    if not inv.get("invoice_number"):
        errors.append("missing_field: invoice_number")

    if not inv.get("invoice_date"):
        errors.append("missing_field: invoice_date")
    else:
        # ---------- Rule 2: Date must be valid ----------
        try:
            parse_date(inv["invoice_date"])
        except:
            errors.append("format_error: invalid_invoice_date")

    # ---------- Rule 3: Amounts must match ----------
    net = inv.get("net_total")
    tax = inv.get("tax_amount")
    gross = inv.get("gross_total")

    if net is not None and tax is not None and gross is not None:
        if abs((net + tax) - gross) > 1:   # Small tolerance
            errors.append("business_rule: totals_mismatch")

    # Result for this invoice
    return {
        "invoice_id": inv.get("invoice_number"),
        "is_valid": len(errors) == 0,
        "errors": errors
    }


# ------------------------------
# Validate ALL invoices
# ------------------------------
def validate_all_invoices(invoices):
    results = []
    error_counts = {}

    for inv in invoices:
        result = validate_invoice(inv)
        results.append(result)

        # Count errors
        for err in result["errors"]:
            error_counts[err] = error_counts.get(err, 0) + 1

    summary = {
        "total_invoices": len(invoices),
        "valid_invoices": sum(r["is_valid"] for r in results),
        "invalid_invoices": sum(not r["is_valid"] for r in results),
        "error_counts": error_counts
    }

    return results, summary


# ------------------------------
# Test validator
# ------------------------------
results, summary = validate_all_invoices(all_invoices)

results, summary


([{'invoice_id': 'AUFNR34343', 'is_valid': True, 'errors': []},
  {'invoice_id': 'AUFNR234953', 'is_valid': True, 'errors': []},
  {'invoice_id': 'AUFNR34869', 'is_valid': True, 'errors': []},
  {'invoice_id': 'AUFNR123456', 'is_valid': True, 'errors': []},
  {'invoice_id': 'AUFNR89493', 'is_valid': True, 'errors': []}],
 {'total_invoices': 5,
  'valid_invoices': 5,
  'invalid_invoices': 0,
  'error_counts': {}})

In [44]:
for inv in all_invoices:
    print(inv["invoice_number"], inv["net_total"], inv["tax_amount"], inv["gross_total"])


AUFNR34343 64.0 None 76.16
AUFNR234953 216.0 None 257.04
AUFNR34869 1080.0 None 1285.2
AUFNR123456 216.0 None 257.04
AUFNR89493 264.0 None 314.16


In [45]:
all_invoices = extract_all_invoices("sample_pdfs")

for inv in all_invoices:
    print(inv["invoice_number"], inv["net_total"], inv["tax_amount"], inv["gross_total"])


Processing: sample_pdfs\sample_pdf_1.pdf
Processing: sample_pdfs\sample_pdf_2.pdf
Processing: sample_pdfs\sample_pdf_3.pdf
Processing: sample_pdfs\sample_pdf_4.pdf
Processing: sample_pdfs\sample_pdf_5.pdf
AUFNR34343 64.0 12.16 76.16
AUFNR234953 216.0 41.04 257.04
AUFNR34869 108.0 205.2 1285.2
AUFNR123456 216.0 41.04 257.04
AUFNR89493 264.0 50.16 314.16


In [46]:
results, summary = validate_all_invoices(all_invoices)

print("VALIDATION RESULTS:")
for r in results:
    print(r)

print("\nSUMMARY:")
print(summary)


VALIDATION RESULTS:
{'invoice_id': 'AUFNR34343', 'is_valid': True, 'errors': []}
{'invoice_id': 'AUFNR234953', 'is_valid': True, 'errors': []}
{'invoice_id': 'AUFNR34869', 'is_valid': False, 'errors': ['business_rule: totals_mismatch']}
{'invoice_id': 'AUFNR123456', 'is_valid': True, 'errors': []}
{'invoice_id': 'AUFNR89493', 'is_valid': True, 'errors': []}

SUMMARY:
{'total_invoices': 5, 'valid_invoices': 4, 'invalid_invoices': 1, 'error_counts': {'business_rule: totals_mismatch': 1}}
